In [62]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

import requests
from langchain_community.tools import DuckDuckGoSearchRun
import os
from dotenv import load_dotenv

load_dotenv()


True

In [63]:
search_tool = DuckDuckGoSearchRun()

# result = search_tool.invoke("capital of india")

# result

In [64]:
@tool
def get_weather(city : str) -> str:
    '''This function give current weather of given city'''
    url = f'http://api.weatherstack.com/current?access_key=5a3fbfb8bc7d3d3475b9983c8ce7dc82&query={city}'
    
    
    response = requests.get(url)
    
    return response.json()

In [65]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash"
)

In [66]:
from langchain_classic.agents import create_react_agent, AgentExecutor

In [67]:
from langchain_core.prompts import PromptTemplate

#creating prompt template due to unable to use prompt from hub

template = """Answer the following question as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: {input}
Thought: You should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

In [68]:
#creating agent -> it handle all logical or reasoning 

agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather],
    prompt=prompt
)

In [69]:
#creating agent executor -> it maintain loop or work flow until final answer

agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather],
    verbose=True,
    handle_parsing_errors=True
)

In [70]:
import os

print("API Key exists:", bool(os.getenv("OPENROUTER_API_KEY")))

API Key exists: True


In [71]:

query = input("How can I assist You ?")

result = agent_executor.invoke({"input" : query})

print(result)

print("AI response : ", result['output'])
    



> Entering new AgentExecutor chain...
Action: get_weather
Action Input: delhi{'request': {'type': 'City', 'query': 'Delhi, Canada', 'language': 'en', 'unit': 'm'}, 'location': {'name': 'Delhi', 'country': 'Canada', 'region': 'Ontario', 'lat': '42.850', 'lon': '-80.500', 'timezone_id': 'America/Toronto', 'localtime': '2026-09-13 14:53', 'localtime_epoch': 1789311180, 'utc_offset': '-4.0'}, 'current': {'observation_time': '06:53 PM', 'temperature': 25, 'weather_code': 176, 'weather_icons': ['https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0009_light_rain_showers.png'], 'weather_descriptions': ['Patchy rain nearby'], 'astro': {'sunrise': '07:00 AM', 'sunset': '07:35 PM', 'moonrise': '09:49 AM', 'moonset': '08:27 PM', 'moon_phase': 'Waxing Crescent', 'moon_illumination': 4}, 'air_quality': {'co': '131', 'no2': '1.1', 'o3': '98', 'so2': '0.9', 'pm2_5': '4', 'pm10': '4.3', 'us-epa-index': '1', 'gb-defra-index': '1'}, 'wind_speed': 22, 'wind_degree': 241, 'wind_dir': 'WS